# Document-Grounded Insurance Chatbot (2025 Academic Prototype)

A research prototype built for a NOVA IMS business case using Azure OpenAI, hosted file search, and retrieval-augmented generation (RAG). **Historical notebook:** this implementation uses the Assistants API and is not expected to run unchanged on current services. The public copy intentionally excludes credentials, source documents, and execution output.


### Import packages

In [ ]:
# Packages loading
import os
import json
import time
import pandas as pd
from openai import AzureOpenAI
from PIL import Image
from IPython.display import Markdown, display
import pickle
from datetime import datetime, timedelta

### Open AI Setup

In [ ]:
# Read Azure OpenAI credentials from environment variables.
# Never paste an API key directly into a notebook or commit credentials to Git.
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
if not api_key or not endpoint:
    raise RuntimeError("Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT in your environment before running.")


In [ ]:
# Constants
current_folder = os.getcwd()
data_folder = current_folder
data_folder_full_path = os.path.abspath(data_folder)

agent_data = 'AgentFiles.pkl'     # Each agent should use their own files

assistantFilename = 'AssistantID.TXT'
assistant_id = None
assistant = None

displayedMessagesIDs = []

In [ ]:
# Initialize the Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint = endpoint,
    api_key= api_key,
    api_version="2024-05-01-preview")

In [ ]:
# Optimized Prompt
aRole = "Atue como um assistente inteligente para os agentes de seguros da Fidelidade, focado em fornecer respostas instantâneas, precisas e claras sobre os produtos “Fidelidade Savings” e “PPR Evoluir”. Ajude os clientes esclarecendo dúvidas sobre a educação financeira e os produtos de seguros. \n Estilo e Tom de Comunicação: Use uma linguagem clara, polida e amigável, que transmita a tradição, confiança e credibilidade da Fidelidade. Garanta respostas objetivas, evitando jargões técnicos sem explicação, para que tanto os agentes quanto os clientes compreendam perfeitamente as informações.\n Mecanismo de Resposta e Fluxo de Conversa: (Identificação do Tema: Inicialmente, identifique se a consulta é sobre detalhes dos produtos ou sobre questões de alfabetização financeira/FAQ. Consulta e Recuperação de Conteúdo (RAG): Ao receber uma pergunta, execute uma consulta ao repositório dos PDFs relativos a 'Fidelidade Savings' e 'PPR Evoluir'. Integre o conteúdo recuperado à resposta de forma coesa e contextualizada, garantindo precisão na informação prestada. Como lidar com incertezas: Se a resposta à consulta não estiver presente ou estiver além do conteúdo treinado, responda de forma transparente: 'Não disponho desta informação no momento. Poderia reformular a sua dúvida?' ou, se for o caso, indique a necessidade de consulta a um especialista humano. Referências e Recursos de Apoio: Sempre que aplicável, sugira links para os documentos ou diretrizes regulatórias, de forma a permitir que o usuário acesse informações complementares e detalhadas.).\n Como lidar com casos complexos: Se detectar que a dúvida é muito complexa ou fora do âmbito da base de conhecimento integrada, oriente o usuário a entrar em contacto com um especialista, mencionando: 'Para uma análise mais aprofundada, recomendo consultar um de nossos especialistas.'.\n Personalização e Continuidade do Diálogo: Mantenha o contexto das conversas anteriores para garantir uma experiência contínua e personalizada. Ofereça sugestões de próximos passos ou perguntas correlatas de forma proativa, incentivando um diálogo mais rico. \n Aprimoramento Contínuo: Aprenda com cada interação, ajustando as suas respostas e sugestões com base no feedback dos usuários. Isso garantirá que você se torne cada vez mais eficaz e relevante para os agentes de seguros da Fidelidade.\n NÃO RESPONDER EM TÓPICOS. NÃO RESPONDER EM TÓPICOS."

In [ ]:
# Optimized Prompt
aRole = "Atue como um assistente inteligente para os agentes de seguros da Fidelidade, focado em fornecer respostas instantâneas, precisas e claras sobre os produtos “Fidelidade Savings” e “PPR Evoluir”. Ajude os clientes esclarecendo dúvidas sobre a educação financeira e os produtos de seguros. \n Estilo e Tom de Comunicação: Use uma linguagem clara, polida e amigável, que transmita a tradição, confiança e credibilidade da Fidelidade. Garanta respostas objetivas, evitando jargões técnicos sem explicação, para que tanto os agentes quanto os clientes compreendam perfeitamente as informações.\n Mecanismo de Resposta e Fluxo de Conversa: (Consulta e Recuperação de Conteúdo (RAG): Ao receber uma pergunta, execute uma consulta ao repositório dos PDFs relativos a 'Fidelidade Savings' e 'PPR Evoluir'. Integre o conteúdo recuperado à resposta de forma coesa e contextualizada, garantindo precisão na informação prestada. Como lidar com incertezas: Se a resposta à consulta não estiver presente ou estiver além do conteúdo treinado, responda de forma transparente: 'Não disponho desta informação no momento. Poderia reformular a sua dúvida?' ou, se for o caso, indique a necessidade de consulta a um especialista humano. Referências e Recursos de Apoio: Sempre que aplicável, sugira links para os documentos ou diretrizes regulatórias, de forma a permitir que o usuário acesse informações complementares e detalhadas.).\n Como lidar com casos complexos: Se detectar que a dúvida é muito complexa ou fora do âmbito da base de conhecimento integrada, oriente o usuário a entrar em contacto com um especialista, mencionando: 'Para uma análise mais aprofundada, recomendo consultar um de nossos especialistas.'.\n Personalização e Continuidade do Diálogo: Mantenha o contexto das conversas anteriores para garantir uma experiência contínua e personalizada. Ofereça sugestões de próximos passos ou perguntas correlatas de forma proativa, incentivando um diálogo mais rico. \n Aprimoramento Contínuo: Aprenda com cada interação, ajustando as suas respostas e sugestões com base no feedback dos usuários. Isso garantirá que você se torne cada vez mais eficaz e relevante para os agentes de seguros da Fidelidade.\n NÃO RESPONDER EM TÓPICOS. NÃO RESPONDER EM TÓPICOS."

### Support functions

In [ ]:
# Function to create the assistant with code interpreter capability
def create_assistant():
    assistant = client.beta.assistants.create(
        name="Agent",
        instructions=aRole,
        model="gpt-4o-mini",
        tools=[{"type": "file_search"}],
        temperature=0.3,
        top_p=0.9
    )
    assistant_id = assistant.id

    # Save the ID to a file
    with open(assistantFilename, "w") as file:
        file.write(assistant_id)

    return assistant

# Function to create a new thread for a conversation
def create_thread():
    thread = client.beta.threads.create()
    displayedMessagesIDs = []
    return thread

# Function to check if the assistant exists
def check_assistant_exists(assistant_id):
    try:
        response = client.beta.assistants.retrieve(assistant_id)
        return True if response else False, response
    except Exception as e:
        print(f"An error occurred while checking the assistant: {e}", assistant_id)
        return False, None

# Function to load files from the current folder and upload them to a vector store
def load_and_upload_files():
    # Only ingest locally supplied PDFs from this explicitly designated folder.
    # Never add private, licensed, or confidential documents to a public repository.
    folder_path = os.path.join(data_folder_full_path, "documents")
    if not os.path.isdir(folder_path):
        print("Create a local documents/ folder and add only PDFs you are authorized to use.")
        return None

    print("Scanning current folder for PDF files...")
    file_paths = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith('.pdf')
    ]
    print("PDF files found:", file_paths)

    if not file_paths:
        print("No PDF files found in the current folder.")
        return None

    file_streams = []
    try:
        file_streams = [open(path, "rb") for path in file_paths]
        print("Opened file streams:", [f.name for f in file_streams])

        # Create a new vector store (API client must be set up already)
        vector_store = client.vector_stores.create(name="PDF Documents")
        print("Vector store created:", vector_store)

        file_batch = client.vector_stores.file_batches.upload_and_poll(
            vector_store_id=vector_store.id,
            files=file_streams
        )
        print("Upload batch result:", file_batch)

        # Poll file statuses until done
        while True:
            files = list(client.vector_stores.files.list(vector_store_id=vector_store.id))
            statuses = [f.status for f in files]
            print("File statuses:", statuses)
            if all(s in ("completed", "failed") for s in statuses) and len(files) == len(file_paths):
                break
            time.sleep(2)

        for f in files:
            print(f"File: {f.id}, Status: {f.status}")

    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None

    for f in file_streams:
        f.close()

    if file_batch.status == "completed" and file_batch.file_counts.completed == len(file_paths):
        print("All files processed successfully. Continuing with assistant...")
    else:
        print("File processing failed. Check Azure portal for details.")

    # Re-fetch vector store to get up-to-date info
    vector_store = client.vector_stores.retrieve(vector_store.id)

    try:
        with open(agent_data, "wb") as file:
            pickle.dump(vector_store, file)
        print(f"Vector store saved to {agent_data}")
    except Exception as e:
        print(f"Error saving vector store: {str(e)}")

    print("Vector store created:", vector_store)
    return vector_store

# Function to add a message to the thread
def add_message_to_thread(thread_id, user_message):
    message = client.beta.threads.messages.create(
        thread_id=thread_id,
        role="user",
        content=user_message
    )
    return message

# Function to download file
def file_download(file_id, thread_id, message_id, file_link='',is_image=True):
    # Identify file and define name to save
    content = client.files.content(file_id)
    if is_image:
        fileName='Images/img_'+thread_id+message_id+'.png'
    else:
        _, file_extension = os.path.splitext(file_link)
        file_base_name = os.path.basename(file_link)
        fileName='OtherFiles/'+file_base_name+thread_id+message_id+file_extension
    # Download file
    file = content.write_to_file(fileName)
    return fileName

# Function to display messages not displayed yet
def display_messages(thread, message):
    messages = client.beta.threads.messages.list(thread_id=thread.id, order='asc', after=message.id)
    if messages:
        for message in messages:
            if message.id not in displayedMessagesIDs:

                # Check if the message is from the assistant and print it
                if message.role == 'assistant':
                    # Load JSON message into a Python object
                    answer = json.loads(message.model_dump_json(indent=2))
                    # Show answer
                    try:
                        messageType = message.content[0].type
                        # Add message id  to the list as displayed
                        displayedMessagesIDs.append(message.id)

                        # Text message
                        if  messageType=='text':
                            content = 'Assistant: ' + answer['content'][0]['text']['value']

                            # Check if there are links for file downloads
                            file_link = None
                            if 'annotations' in answer['content'][0]['text']:
                                for annotation in answer['content'][0]['text']['annotations']:
                                    if annotation['type'] == 'file_path':
                                        file_link = annotation['text']
                                        file_id = annotation['file_path'].get('file_id')
                                        start_index = annotation.get('start_index')
                                        end_index = annotation.get('end_index')

                                        markdown_link = f"[Ver documento]({file_link})"
                                        # Replace text of annotation by link
                                        if start_index is not None and end_index is not None:
                                            content = content[:start_index] + markdown_link + content[end_index:]

                                        # Download the file
                                        fileName = file_download(file_id, thread.id, message.id, file_link=file_link,is_image=False)

                            # Display as Markdown
                            display(Markdown(content))
                        # Image
                        elif messageType == 'image_file':
                            # Get the ID of the image
                            fileID = answer['content'][0]['image_file']['file_id']

                            # Download the image
                            fileName = file_download(fileID, thread.id, message.id, file_link='', is_image=True)

                            # Display the image in the default image viewer
                            image = Image.open(fileName)
                            image.show()
                    except:
                        continue

# Function to send a message to the assistant and get a response
def send_message_to_assistant(thread, user_input):
    # Present feedback to user
    print("Tu:", user_input)
    print("Pensando...")

    # Add message to thread
    message = add_message_to_thread(thread.id, user_input)

    # Run the thread with the assistant
    run = client.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id
    )

    # Loop until the run completes or fails
    while run.status in ['queued', 'in_progress', 'cancelling']:
        time.sleep(1)
        run = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id
        )
        # Display any messages that are being become available
        try:
            display_messages(thread, message)
        except:
            continue

    # Handle the response from the assistant
    if run.status == 'completed':
        # Display the messages
        display_messages(thread, message)

    elif run.status == 'requires_action':
        print("The assistant requires additional actions.")
    else:
        print(f"Run status: {run.status}")

### Main

In [ ]:
# Check existent assistants
if os.path.exists(assistantFilename):
    with open(assistantFilename, "r") as file:
        assistant_id = file.read()

    # Check on ChatGPT
    exists, assistant = check_assistant_exists(assistant_id)
    if exists:
        # Get the vector_store
        with open(agent_data, "rb") as file:  # Use binary mode to read
            vector_store = pickle.load(file)

        assistant = client.beta.assistants.update(
            assistant_id=assistant_id,
            tool_resources={
                            "file_search": {
                            "vector_store_ids": [vector_store.id]
                            }
    }
)

        # Provide feedback
        print("Using assistant ", assistant_id)
    else:
        assistant_id = None

In [ ]:
# Create assistant if one does not exists - Load the data
if assistant_id == None:
    # Create assistant
    print("Creating new assistant")
    assistant = create_assistant()
    assistant_id = assistant.id

    # Load files from the local folder and upload to vector store
    vector_store = load_and_upload_files()
    print("Files uploaded.")

    # Upload the assistant with the new files
    assistant = client.beta.assistants.update(
        assistant_id=assistant.id,
        tool_resources = {"file_search": {"vector_store_ids": [vector_store.id]}}
    )

In [ ]:
# New thread
thread = create_thread()

#### Main Loop

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.strip().upper() == "QUIT":
        print("Saindo. Adeus!")
        break
    elif user_input.strip().upper() == "RESET":
        print("Começando uma nova conversa...")
        thread = create_thread()
    else:
        send_message_to_assistant(thread, user_input)

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.strip().upper() == "QUIT":
        print("Saindo. Adeus!")
        break
    elif user_input.strip().upper() == "RESET":
        print("Começando uma nova conversa...")
        thread = create_thread()
    else:
        send_message_to_assistant(thread, user_input)